# PROJETO CARDS FIFA — POO 2026
**Calebe Soares Alencar e Rafael Andrade Morais**

---
Cada célula representa um passo/conceito do projeto.

## Passo 1 — Classe base abstrata `Jogador`
A classe mãe define os atributos **comuns** a todo jogador e obriga as subclasses a implementarem `calcular_overall`.

In [ ]:
from abc import ABC, abstractmethod

class Jogador(ABC):
    # Atributos comuns a qualquer posição
    def __init__(self, nome, idade, nacionalidade):
        self.nome = nome
        self.idade = idade
        self.nacionalidade = nacionalidade

    # Método abstrato: cada posição DEVE implementar sua própria fórmula
    @abstractmethod
    def calcular_overall(self):
        pass

    # Representação em texto do card — usa o overall calculado pela subclasse
    def __str__(self):
        overall = self.calcular_overall()
        posicao = type(self).__name__  # pega o nome da classe (ex: 'Atacante')
        return (
            f"\n{'='*30}\n"
            f"  {overall}  {posicao.upper()}\n"
            f"  {self.nome}\n"
            f"  {self.nacionalidade} | {self.idade} anos\n"
            f"{'='*30}"
        )

## Passo 2 — Subclasse `Atacante`
Herda de `Jogador` e implementa `calcular_overall` com **pesos ofensivos**.

| Atributo | Peso |
|----------|------|
| Chute    | 25%  |
| Drible   | 20%  |
| Velocidade | 20% |
| Passe    | 15%  |
| Físico   | 15%  |
| Defesa   | 5%   |

In [ ]:
class Atacante(Jogador):
    def __init__(self, nome, idade, nacionalidade, velocidade, chute, passe, drible, defesa, fisico):
        # Chama o construtor da classe mãe para os atributos comuns
        super().__init__(nome, idade, nacionalidade)
        # Atributos específicos da posição
        self.velocidade = velocidade
        self.chute = chute
        self.passe = passe
        self.drible = drible
        self.defesa = defesa
        self.fisico = fisico

    def calcular_overall(self):
        # Fórmula ponderada: atributos ofensivos valem mais para um atacante
        overall = (
            self.chute      * 0.25 +
            self.drible     * 0.20 +
            self.velocidade * 0.20 +
            self.passe      * 0.15 +
            self.fisico     * 0.15 +
            self.defesa     * 0.05
        )
        return int(overall)  # arredonda para baixo, como no FIFA

## Passo 3 — Subclasse `Goleiro`
Goleiros têm atributos **completamente diferentes** — isso mostra por que herança é útil.

In [ ]:
class Goleiro(Jogador):
    def __init__(self, nome, idade, nacionalidade, elasticidade, manejo, chute, reflexo, posicionamento, velocidade):
        super().__init__(nome, idade, nacionalidade)
        self.elasticidade    = elasticidade
        self.manejo          = manejo
        self.chute           = chute      # chute de meta
        self.reflexo         = reflexo
        self.posicionamento  = posicionamento
        self.velocidade      = velocidade

    def calcular_overall(self):
        # Para goleiro, reflexo e posicionamento pesam muito mais
        overall = (
            self.reflexo        * 0.30 +
            self.posicionamento * 0.25 +
            self.elasticidade   * 0.20 +
            self.manejo         * 0.15 +
            self.velocidade     * 0.05 +
            self.chute          * 0.05
        )
        return int(overall)

## Passo 4 — Subclasse `Zagueiro`
Praticando o padrão: herança + fórmula própria.

In [ ]:
class Zagueiro(Jogador):
    def __init__(self, nome, idade, nacionalidade, velocidade, chute, passe, drible, defesa, fisico):
        super().__init__(nome, idade, nacionalidade)
        self.velocidade = velocidade
        self.chute      = chute
        self.passe      = passe
        self.drible     = drible
        self.defesa     = defesa
        self.fisico     = fisico

    def calcular_overall(self):
        # Para zagueiro, defesa e físico são os atributos mais importantes
        overall = (
            self.defesa     * 0.35 +
            self.fisico     * 0.25 +
            self.velocidade * 0.15 +
            self.passe      * 0.15 +
            self.chute      * 0.05 +
            self.drible     * 0.05
        )
        return int(overall)

## Passo 5 — Comparar jogadores
Função que recebe **qualquer** lista de `Jogador` e exibe os cards ordenados pelo overall.

In [ ]:
def comparar_jogadores(jogadores):
    # Ordena usando o método de cada objeto — polimorfismo em ação
    ordenados = sorted(jogadores, key=lambda j: j.calcular_overall(), reverse=True)

    print("\n=== COMPARAÇÃO DE JOGADORES ===")
    for posicao, jogador in enumerate(ordenados, start=1):
        print(f"\n#{posicao} Overall: {jogador.calcular_overall()}")
        print(jogador)

---
## Etapa 1 — Arquivo de dados `jogadores.json`

Até aqui os jogadores eram criados **na mão** dentro do código. Isso é ruim porque:
- Para adicionar um jogador novo, você precisa mexer no código
- Dados misturados com lógica é difícil de manter

A solução: separar os **dados** (JSON) da **lógica** (classes Python).

### Estrutura do `jogadores.json`
Cada jogador é um objeto JSON com os campos que a classe correspondente precisa.
O campo `"posicao"` é especial — ele vai dizer qual classe instanciar.

```json
[
  {
    "nome": "Vinicius Jr",
    "posicao": "Atacante",
    "nacionalidade": "Brasil",
    "idade": 24,
    "velocidade": 95,
    "chute": 87,
    ...
  }
]
```

> **Atenção:** Goleiro tem atributos diferentes dos demais (elasticidade, reflexo...).  
> Por isso cada posição tem seu próprio conjunto de campos no JSON.

In [ ]:
import json

# Abre e lê o arquivo — json.load converte o texto em lista de dicionários Python
with open("jogadores.json", encoding="utf-8") as arquivo:
    dados = json.load(arquivo)

# Veja o que foi carregado: uma lista onde cada item é um dicionário
print(f"Total de jogadores no arquivo: {len(dados)}")
print("\nPrimeiro jogador (como dicionário Python):")
print(dados[0])

### Por que `with open(...)`?
O `with` garante que o arquivo seja **fechado automaticamente** após a leitura, mesmo que ocorra um erro. Sem ele, você precisaria chamar `arquivo.close()` manualmente.

### O que é `json.load`?
Transforma o texto do arquivo em estruturas Python:
- `{ }` do JSON → `dict` do Python  
- `[ ]` do JSON → `list` do Python  
- strings, números → tipos Python equivalentes

In [ ]:
# Explorando os dados carregados
print("Jogadores no arquivo:\n")
for jogador in dados:
    # Acessamos os campos do dicionário com jogador["campo"]
    print(f"  {jogador['nome']:20s} | {jogador['posicao']:10s} | {jogador['nacionalidade']}")

---
## Etapa 2 — Função fábrica `criar_jogador`

Agora precisamos transformar cada dicionário em um **objeto** da classe certa.  
Para isso, criamos uma **função fábrica**: ela recebe um dicionário e devolve a instância correta.

### Conceitos novos aqui:
- `**dados` — desempacota um dicionário como argumentos nomeados
- `dados.pop("chave")` — remove e retorna o valor de uma chave
- Dicionário como mapa de classes

In [ ]:
def criar_jogador(dados):
    # Copia para não modificar o dicionário original
    d = dados.copy()

    # Remove 'posicao' do dicionário — ela não é um parâmetro do __init__
    posicao = d.pop("posicao")

    # Mapeia o nome da posição para a classe correspondente
    classes = {
        "Atacante": Atacante,
        "Goleiro":  Goleiro,
        "Zagueiro": Zagueiro,
    }

    # Verifica se a posição existe no mapa
    if posicao not in classes:
        raise ValueError(f"Posição desconhecida: {posicao}")

    # **d desempacota o dicionário como argumentos nomeados
    # É equivalente a: Atacante(nome="...", idade=24, nacionalidade="...", ...)
    return classes[posicao](**d)


# Testando com o primeiro jogador do arquivo
primeiro = criar_jogador(dados[0])
print(primeiro)
print(f"Tipo do objeto: {type(primeiro)}")

---
## Etapa 3 — Carregar todos os jogadores e exibir os cards

In [ ]:
# Cria um objeto para cada entrada do JSON usando a função fábrica
elenco = [criar_jogador(d) for d in dados]

# Exibe o card de cada jogador
for jogador in elenco:
    print(jogador)

In [ ]:
# Comparação geral do elenco carregado do JSON
comparar_jogadores(elenco)